# ✝️ Gospel Lyric Video — *Jesus Comin' By and By*

**Run each cell top to bottom. Tap ▶ or press Shift+Enter on each one.**

| Cell | What it does | Est. time |
|---|---|---|
| 1 | Install libraries | ~1 min |
| 2 | (Optional) Upload your MP3 | instant |
| 3 | Build the 3-min video | ~3–5 min |
| 4 | Download gospel_video.mp4 | instant |

> No desktop needed — runs 100% in your browser!

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Install libraries
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip install -q moviepy Pillow numpy
print('✅ Libraries installed!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — (OPTIONAL) Upload your song audio
#   Skip this cell for a silent video.
#   Supported: MP3, WAV, M4A
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import files
import os

AUDIO_FILE = None

print('Choose your audio file (MP3/WAV/M4A)…')
uploaded = files.upload()

if uploaded:
    AUDIO_FILE = list(uploaded.keys())[0]
    size_mb = os.path.getsize(AUDIO_FILE) / 1_000_000
    print(f'✅ Audio ready: {AUDIO_FILE}  ({size_mb:.1f} MB)')
else:
    print('⏭  No file uploaded — video will be silent.')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Build the gospel lyric video
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os
from moviepy import TextClip, CompositeVideoClip, VideoClip, AudioFileClip
from moviepy.video.fx import CrossFadeIn, CrossFadeOut
import numpy as np
from PIL import Image, ImageDraw, ImageFilter

# ── Settings ────────────────────────────────
TARGET = 180        # 3 minutes exactly
W, H   = 1920, 1080
FPS    = 24
OUTPUT = 'gospel_video.mp4'

# ── Lyrics (start_sec, end_sec, text, section) ────────
SECTIONS = [
    (0,   6,   '',                                                           'intro'),
    # Verse 1
    (6,   12,  "Last I saw that star shine yonder in the sky,",              'verse'),
    (12,  18,  "Burnin' bright like a promise that will never die,",         'verse'),
    (18,  24,  "Pointed straight to a Savior born for you and I,",           'verse'),
    (24,  32,  "Oh I felt His Spirit whisper,\n\"He's comin' by and by.\"",  'verse'),
    # Verse 2
    (32,  38,  "Through the trials and the valleys, through the tears I've cried,", 'verse'),
    (38,  44,  "There's a hope that keeps me steady deep inside,",           'verse'),
    (44,  50,  "Ain't no grave gonna hold me, ain't no shadow gonna hide,",  'verse'),
    (50,  58,  "When I hear that trumpet sound,\nHe's comin' by and by.",    'verse'),
    # Chorus 1
    (58,  63,  "Hallelujah!\nHe's comin' for the ready,",                    'chorus'),
    (63,  67,  "Lift your voice and testify!",                               'chorus'),
    (67,  72,  "Saints are shoutin', hearts are steady,",                    'chorus'),
    (72,  78,  "Jesus Christ is comin' by and by!",                          'chorus'),
    (78,  83,  "Oh the heavens will open wide,",                             'chorus'),
    (83,  88,  "And we'll meet Him in the sky,",                             'chorus'),
    (88,  96,  "Hallelujah! Hallelujah!\nJesus comin' by and by!",           'chorus'),
    # Verse 3
    (96,  102, "I can see that eastern sky begin to glow,",                  'verse'),
    (102, 108, "Feel the fire of revival in my soul,",                       'verse'),
    (108, 114, "Every burden gonna vanish, every knee will bow,",            'verse'),
    (114, 122, "And the King of all creation's\ncomin' for me now.",         'verse'),
    # Bridge
    (122, 128, "Oh can't you hear the angels singin'?",                      'bridge'),
    (128, 134, "Oh can't you feel redemption ringin'?",                      'bridge'),
    (134, 142, "Every chain is breakin',\nevery heart awakenin',",           'bridge'),
    (142, 150, "Glory to the Lamb!",                                         'bridge'),
    # Chorus 2
    (150, 155, "Hallelujah!\nHe's comin' for the ready,",                    'chorus'),
    (155, 159, "Lift your voice and testify!",                               'chorus'),
    (159, 164, "Saints are shoutin', hearts are steady,",                    'chorus'),
    (164, 170, "Jesus Christ is comin' by and by!",                          'chorus'),
    (170, 175, "Oh the heavens will open wide,",                             'chorus'),
    (175, 179, "And we'll meet Him in the sky,",                             'chorus'),
    (179, 184, "Hallelujah! Hallelujah!\nJesus comin' by and by!",           'chorus'),
    # Outro
    (184, 188, "By and by\u2026  (He's comin')",                             'outro'),
    (188, 192, "By and by\u2026  (Oh yes He is)",                            'outro'),
    (192, 198, "Hallelujah, hallelujah,",                                    'outro'),
    (198, 207, "Jesus comin' by and by!",                                    'outro'),
    (207, 213, '',                                                           'intro'),
]

SECTION_COLORS = {
    'verse':  (255, 240, 200),
    'chorus': (255, 220,  50),
    'bridge': (200, 230, 255),
    'outro':  (255, 200, 120),
    'intro':  (255, 255, 255),
}

# ── Star field (pre-rendered, static) ───────────────
def make_stars():
    img  = Image.new('RGBA', (W, H), (0,0,0,0))
    draw = ImageDraw.Draw(img)
    rng  = np.random.default_rng(42)
    n    = 400
    xs   = rng.integers(0, W, n)
    ys   = rng.integers(0, int(H*0.65), n)
    rs   = rng.choice([1,1,1,2,2,3], size=n)
    als  = rng.integers(100, 255, n)
    for x, y, r, a in zip(xs, ys, rs, als):
        draw.ellipse([x-r, y-r, x+r, y+r], fill=(255,255,240,int(a)))
    return np.array(img)

# ── Glowing cross (pre-rendered, right side) ──────────
def make_cross():
    img  = Image.new('RGBA', (W, H), (0,0,0,0))
    draw = ImageDraw.Draw(img)
    cx, cy, t = int(W*0.82), int(H*0.40), 20
    draw.rectangle([cx-t//2, cy-100, cx+t//2, cy+100], fill=(255,230,120,55))
    draw.rectangle([cx-90,   cy-t//2, cx+90,  cy+t//2], fill=(255,230,120,55))
    pil = Image.fromarray(np.array(img)).filter(ImageFilter.GaussianBlur(20))
    return np.array(pil)

# ── Horizon glow (golden light at bottom) ─────────────
def make_horizon():
    img  = Image.new('RGBA', (W, H), (0,0,0,0))
    draw = ImageDraw.Draw(img)
    for i in range(120):
        alpha = int(60 * (1 - i/120))
        y = H - 1 - i
        draw.line([(0,y),(W,y)], fill=(255,200,80,alpha))
    pil = Image.fromarray(np.array(img)).filter(ImageFilter.GaussianBlur(8))
    return np.array(pil)

print('Pre-rendering overlays…')
_stars   = make_stars()
_cross   = make_cross()
_horizon = make_horizon()

def blend(base, overlay):
    a   = overlay[:,:,3:4].astype(float) / 255.0
    rgb = overlay[:,:,:3].astype(float)
    return np.clip(base.astype(float)*(1-a) + rgb*a, 0, 255).astype(np.uint8)

def bg_frame(t):
    # Gradient sky — slowly brightens as the song progresses
    phase = t / TARGET
    top   = np.clip([10+phase*8,  18+phase*20, 60+phase*40],  0, 255)
    bot   = np.clip([40+phase*30, 10+phase*5,  80+phase*20],  0, 255)
    rows  = np.linspace(0, 1, H)[:, None]
    frame = (top*(1-rows) + bot*rows).astype(np.uint8)
    frame = np.broadcast_to(frame, (H, W, 3)).copy()

    # Twinkling stars: gentle global pulse on overlay alpha
    star_mod = _stars.astype(float).copy()
    twinkle  = 0.75 + 0.25 * np.sin(t * 1.8 + 0.5)
    star_mod[:,:,3] = np.clip(star_mod[:,:,3] * twinkle, 0, 255)
    frame = blend(frame, star_mod.astype(np.uint8))

    frame = blend(frame, _horizon)
    frame = blend(frame, _cross)
    return frame

# ── Lyric clip factory ───────────────────────────
def make_lyric(start, end, text, section):
    start, end = min(start, TARGET), min(end, TARGET)
    dur = end - start
    if dur <= 0 or not text.strip():
        return None
    hex_col = '#{:02X}{:02X}{:02X}'.format(*SECTION_COLORS.get(section, (255,255,255)))
    mlen    = max(len(l) for l in text.split('\n'))
    fsize   = 72 if mlen <= 30 else 60 if mlen <= 45 else 50
    clip = TextClip(
        text=text, font_size=fsize, color=hex_col,
        font='DejaVu-Sans-Bold', text_align='center',
        method='caption', size=(int(W*0.85), None),
        stroke_color='#000000', stroke_width=3,
    )
    fade = min(0.7, dur * 0.22)
    clip = clip.with_effects([CrossFadeIn(fade), CrossFadeOut(fade)])
    clip = clip.with_duration(dur).with_start(start)
    y = {'chorus': int(H*0.36), 'bridge': int(H*0.42),
         'outro':  int(H*0.48)}.get(section, int(H*0.58))
    return clip.with_position(('center', y))

# ── Section label factory ────────────────────────
def make_label(text, start, end):
    start, end = min(start, TARGET), min(end, TARGET)
    dur = end - start
    if dur <= 0: return None
    clip = TextClip(text=text, font_size=26, color='#99AACC',
                    font='DejaVu-Sans', method='label')
    clip = clip.with_effects([CrossFadeIn(0.5), CrossFadeOut(0.5)])
    return clip.with_duration(dur).with_start(start).with_position((55, 38))

# ── Build everything ────────────────────────────
print('Building animated background…')
bg = VideoClip(bg_frame, duration=TARGET).with_fps(FPS)

print('Building title card…')
title = (TextClip(text="Jesus Comin' By and By", font_size=92,
                  color='#FFD700', font='DejaVu-Sans-Bold', method='label',
                  stroke_color='#000000', stroke_width=4)
         .with_duration(6).with_start(0)
         .with_position(('center', int(H*0.37)))
         .with_effects([CrossFadeIn(1.2), CrossFadeOut(1.0)]))

subtitle = (TextClip(text='A Gospel Celebration', font_size=46,
                     color='#FFFFFF', font='DejaVu-Sans', method='label',
                     stroke_color='#000000', stroke_width=2)
            .with_duration(6).with_start(0)
            .with_position(('center', int(H*0.53)))
            .with_effects([CrossFadeIn(1.8), CrossFadeOut(1.0)]))

print('Building lyric clips…')
lyrics = [c for s in SECTIONS if (c := make_lyric(*s))]

labels = [
    make_label('♪ Verse 1',  6,   32),
    make_label('♪ Verse 2',  32,  58),
    make_label('✦ Chorus',   58,  96),
    make_label('♪ Verse 3',  96,  122),
    make_label('♩ Bridge',   122, 150),
    make_label('✦ Chorus',   150, 184),
    make_label('♪ Outro',    184, TARGET),
]
labels = [l for l in labels if l]

print('Compositing final video…')
final = CompositeVideoClip(
    [bg, title, subtitle] + labels + lyrics,
    size=(W, H)
).with_duration(TARGET).with_fps(FPS)

# ── Attach audio if uploaded in Cell 2 ───────────────
try:
    if AUDIO_FILE and os.path.exists(AUDIO_FILE):
        print(f'Attaching audio: {AUDIO_FILE}…')
        audio = AudioFileClip(AUDIO_FILE)
        if audio.duration > TARGET:
            audio = audio.subclipped(0, TARGET)
        final = final.with_audio(audio)
        print('✅ Audio attached!')
    else:
        print('ℹ️  No audio — rendering silent video.')
except NameError:
    print('ℹ️  Cell 2 was skipped — rendering silent video.')

# ── Render ────────────────────────────────────
print('\n🎬 Rendering — please wait 3–5 minutes…')
final.write_videofile(
    OUTPUT,
    fps=FPS,
    codec='libx264',
    audio_codec='aac',
    preset='fast',
    ffmpeg_params=['-crf', '23'],
    logger='bar',
)
print(f'\n✅ Done!  →  {OUTPUT}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Download the finished video to your phone
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os
from google.colab import files

OUTPUT = 'gospel_video.mp4'
size_mb = os.path.getsize(OUTPUT) / 1_000_000
print(f'File size: {size_mb:.1f} MB')
print('Starting download…')
files.download(OUTPUT)
print('✅ Check your Downloads folder!')